# pipe_catedra/03 — Optuna: busqueda bayesiana (porteo de `optuna_3.ipynb` / z303)

Input  : `z302_features_{modo}.parquet`
Output : `z303_hiper_{modo}_{experimento}.json`

Porteo fiel del notebook de la catedra -- misma logica, mismas palancas.
Sin loops de Python que valga la pena reescribir con DuckDB (Optuna llama
`objective()` trial por trial, cada uno entrena un LightGBM -- el costo
esta ahi, no en la preparacion de datos).

Responsabilidades:
- Espacio de busqueda de LightGBM con Optuna (TPE = Bayesiana)
- Validacion temporal sin leakage (esquemas `ultimo_periodo` / `walk_forward_k` / `febreros`)
- Features categoricas declaradas a LGBM
- WAPE siempre medido sobre el NIVEL en toneladas (reconstruye si target=delta)
- `tipo_target` se lee del dataset y se propaga al JSON
- Storage SQLite (trials acumulables, resumible)


## 0) Setup


In [ ]:
import os, shutil, time
from pathlib import Path

import polars as pl
import numpy as np
import lightgbm as lgb
import optuna
import json
from tqdm.auto import tqdm

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    """VM de la catedra -> ~/buckets/b1 | Colab -> /content/buckets/b1 | local -> env."""
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1",
                 "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError(
        "No encontre el bucket. Defini LABO3_BUCKET, ej: "
        "os.environ['LABO3_BUCKET'] = '/home/usuario/labo3-bucket'"
    )


BUCKET  = resolver_bucket()
DIR_OUT = BUCKET / "exp_pipe_catedra"
DIR_OUT.mkdir(parents=True, exist_ok=True)
print(f"BUCKET: {BUCKET}")
print(f"salida: {DIR_OUT}")


## 1) Parametros — palancas


In [ ]:
PARAM = {
    'experimento': 'z303',

    # PALANCA 1 (debe coincidir con 01_/02_)
    'modo_agrupacion': 'producto',

    # PALANCA 8: trials nuevos por corrida
    'n_trials': 50,

    # PALANCA 9: esquema de validacion
    # 'ultimo_periodo' -> valida en el ultimo mes con target conocido
    # 'walk_forward_k' -> k splits consecutivos hacia atras
    # 'febreros'       -> valida en los febreros del historico (representa mejor feb 2020)
    'esquema_val': 'febreros',
    'walk_forward_k': 3,
    'mes_objetivo': 2,   # mes que se quiere predecir (2=febrero). El horizonte ya se tiene en cuenta.

    # PALANCA 10: metrica
    'metrica': 'wape',

    # PALANCA 11: sampling de filas
    'sampling_frac': None,

    # PALANCA 12: objetivo de LGBM
    # 'regression' | 'tweedie' | 'poisson' | 'regression_l1'
    'objective_lgbm': 'regression',
    'tweedie_optimizar': True,

    # PALANCA 15: regularizacion
    # 'normal' -> rangos amplios.  'fuerte' -> arboles chicos, mas regularizacion
    'regularizacion': 'normal',

    # PALANCA 16: peso por recencia
    # None -> todos los periodos pesan igual
    # float -> decay (0.9 = cada mes hacia atras pesa 0.9x el siguiente)
    'decay_recencia': None,

    # PALANCA 5: tipo de target (elige la columna)
    # 'nivel' -> usa target_nivel.   'delta' -> usa target_delta
    'tipo_target': 'nivel',

    # PALANCA 19: features a EXCLUIR
    # Por defecto se usan TODAS las features del dataset.
    # Poné aca los nombres de columnas que queres sacar en este experimento.
    'features_excluir': [],

    'semilla': 102191,

    'cols_categoricas': ['cat1', 'cat2', 'cat3', 'brand'],
}

MODO = PARAM['modo_agrupacion']
PARAM['path_input']   = str(DIR_OUT / f"z302_features_{MODO}.parquet")
PARAM['path_output']  = str(DIR_OUT / f"z303_hiper_{MODO}_{PARAM['experimento']}.json")
PARAM['path_storage'] = f"sqlite:///{Path.home() / f'z303_optuna_{MODO}.db'}"
PARAM['study_name']   = f"{PARAM['experimento']}_{MODO}"

print('Parametros:', PARAM)
print('Input:',  PARAM['path_input'])
print('Study:',  PARAM['study_name'])


## 2) Carga y features

Lee `tipo_target` del dataset (lo propaga `02_`). `tn` y `tipo_target` no son features pero se conservan en `df_pd` para reconstruir el nivel.


In [ ]:
df = pl.read_parquet(PARAM['path_input'])
print(f'Dataset: {df.shape}')

TIPO_TARGET = PARAM['tipo_target']
TARGET_COL  = 'target_delta' if TIPO_TARGET == 'delta' else 'target_nivel'
print(f'Tipo de target: {TIPO_TARGET}  (columna: {TARGET_COL})')

COLS_EXCLUIR_BASE = [
    'agrupa_id', 'product_id', 'customer_id', 'periodo',
    'tn', 'tn_t2', 'target_nivel', 'target_delta',
    'modo_agrupacion', 'solo_predecir', 'tipo_target',
]

FEATURES = [c for c in df.columns
            if c not in COLS_EXCLUIR_BASE and c not in PARAM['features_excluir']]
CAT_FEATURES = [c for c in PARAM['cols_categoricas'] if c in FEATURES]

print(f'Features usadas ({len(FEATURES)}): {FEATURES}')
if PARAM['features_excluir']:
    print(f'Features excluidas: {PARAM["features_excluir"]}')
print(f'Categoricas: {CAT_FEATURES}')


## 3) Metrica

WAPE sobre niveles en toneladas. Recibe `y_real_nivel` y `y_pred_nivel` ya reconstruidos.


In [ ]:
def calcular_metrica(y_real, y_pred, metrica='wape'):
    y_real = np.array(y_real, dtype=np.float64)
    y_pred = np.maximum(np.array(y_pred, dtype=np.float64), 0.0)
    if metrica == 'wape':
        den = y_real.sum()
        return np.nan if den == 0 else np.abs(y_real - y_pred).sum() / den
    elif metrica == 'mae':
        return np.abs(y_real - y_pred).mean()
    raise ValueError(f'Metrica desconocida: {metrica}')


## 4) Validacion temporal


In [ ]:
periodos_ordenados = sorted(df['periodo'].unique().to_list())
print(f'Periodos: {periodos_ordenados[0]} -> {periodos_ordenados[-1]}')

# Horizonte de prediccion (debe coincidir con 02_). Se predice t+H.
HORIZONTE = 2

def sumar_meses(periodo, n):
    """Suma n meses a un periodo YYYYMM."""
    anio, mes = divmod(periodo, 100)
    total = (anio * 12 + (mes - 1)) + n
    return (total // 12) * 100 + (total % 12) + 1

def get_splits(periodos, esquema, k=3, mes_obj=2):
    if esquema == 'ultimo_periodo':
        return [(periodos[-2], periodos[-1])]

    elif esquema == 'walk_forward_k':
        splits = []
        for i in range(k, 0, -1):
            splits.append((periodos[-(i + 1)], periodos[-i]))
        return splits

    elif esquema == 'febreros':
        # Queremos validar la prediccion del mes objetivo (febrero).
        # Como se predice t+HORIZONTE, la fila de PARTIDA que predice un febrero
        # esta HORIZONTE meses antes (diciembre, si mes_obj=2 y H=2).
        # val_p = periodo de partida cuyo target cae en el mes objetivo.
        pset = set(periodos)
        splits = []
        for p in periodos:
            objetivo = sumar_meses(p, HORIZONTE)
            if objetivo % 100 == mes_obj and objetivo in pset:
                splits.append((p, p))   # entrena con <= p, valida en la fila p
        if not splits:
            print('aviso: No se pudo armar validacion por febreros; usando ultimo_periodo')
            return [(periodos[-2], periodos[-1])]
        return splits

    raise ValueError(f'Esquema desconocido: {esquema}')

splits = get_splits(periodos_ordenados, PARAM['esquema_val'],
                    PARAM['walk_forward_k'], PARAM.get('mes_objetivo', 2))
print(f'Splits de validacion ({PARAM["esquema_val"]}):')
for corte, val_p in splits:
    objetivo = sumar_meses(val_p, HORIZONTE)
    print(f'  Train <= {corte}  |  Val (fila) = {val_p}  ->  predice {objetivo}')


## 5) Funcion objetivo

Reconstruccion del nivel:
- target='nivel' -> pred_nivel = pred,  real_nivel = target
- target='delta' -> pred_nivel = tn + pred,  real_nivel = tn + target  (= tn_t2)

El WAPE se calcula siempre sobre el nivel, igual que Kaggle.


In [ ]:
df_pd = df.to_pandas()
for c in CAT_FEATURES:
    df_pd[c] = df_pd[c].astype('category')

def calcular_pesos(periodos_serie, decay):
    """Peso por recencia: el periodo mas reciente pesa 1, cada mes hacia atras decae."""
    if decay is None:
        return None
    periodos = sorted(periodos_serie.unique())
    idx = {p: i for i, p in enumerate(periodos)}
    n = len(periodos)
    return periodos_serie.map(lambda p: decay ** (n - 1 - idx[p])).values

def espacio_hiper(trial):
    """Rangos de hiperparametros segun la palanca de regularizacion."""
    base = {
        'objective':     PARAM['objective_lgbm'],
        'metric':        'mae',
        'verbosity':     -1,
        'boosting_type': 'gbdt',
        'seed':          PARAM['semilla'],
        'subsample_freq':1,
    }
    if PARAM['regularizacion'] == 'fuerte':
        base.update({
            'num_leaves':       trial.suggest_int('num_leaves', 8, 64),
            'max_depth':        trial.suggest_int('max_depth', 3, 7),
            'learning_rate':    trial.suggest_float('learning_rate', 5e-3, 0.1, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 800),
            'min_child_samples':trial.suggest_int('min_child_samples', 30, 200),
            'subsample':        trial.suggest_float('subsample', 0.5, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
            'reg_alpha':        trial.suggest_float('reg_alpha', 0.1, 20.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 0.1, 20.0, log=True),
        })
    else:  # 'normal'
        base.update({
            'num_leaves':       trial.suggest_int('num_leaves', 20, 300),
            'max_depth':        trial.suggest_int('max_depth', 3, 12),
            'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
            'n_estimators':     trial.suggest_int('n_estimators', 100, 2000),
            'min_child_samples':trial.suggest_int('min_child_samples', 5, 100),
            'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        })
    if PARAM['objective_lgbm'] == 'tweedie' and PARAM.get('tweedie_optimizar', False):
        base['tweedie_variance_power'] = trial.suggest_float('tweedie_variance_power', 1.1, 1.9)
    return base

def objective(trial):
    params = espacio_hiper(trial)

    errores = []
    for corte, val_p in splits:
        # Train estrictamente anterior a la fila de validacion (evita leakage
        # cuando corte == val_p, como en el esquema 'febreros')
        df_tr = df_pd[df_pd['periodo'] < val_p].copy()
        df_vl = df_pd[df_pd['periodo'] == val_p].copy()
        if len(df_vl) == 0:
            continue
        if PARAM['sampling_frac'] is not None:
            df_tr = df_tr.sample(frac=PARAM['sampling_frac'], random_state=PARAM['semilla'])

        X_tr, y_tr = df_tr[FEATURES], df_tr[TARGET_COL].values
        X_vl, y_vl = df_vl[FEATURES], df_vl[TARGET_COL].values
        w_tr = calcular_pesos(df_tr['periodo'], PARAM['decay_recencia'])

        modelo = lgb.LGBMRegressor(**params)
        modelo.fit(
            X_tr, y_tr,
            sample_weight=w_tr,
            eval_set=[(X_vl, y_vl)],
            categorical_feature=CAT_FEATURES,
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
        )
        pred = modelo.predict(X_vl)

        if TIPO_TARGET == 'delta':
            tn_actual  = df_vl['tn'].values
            pred_nivel = tn_actual + pred
            real_nivel = tn_actual + y_vl
        else:
            pred_nivel = pred
            real_nivel = y_vl

        errores.append(calcular_metrica(real_nivel, pred_nivel, PARAM['metrica']))

    return float(np.mean(errores))


## 6) Correr Optuna (storage persistente)

Los trials se acumulan en SQLite. Para empezar limpio, cambia `experimento` en PARAM.


In [ ]:
study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
    study_name=PARAM['study_name'],
    storage=PARAM['path_storage'],
    load_if_exists=True
)

print(f'Trials previos: {len(study.trials)}')
print(f'Corriendo {PARAM["n_trials"]} trials nuevos...')

with tqdm(total=PARAM['n_trials'], desc='Optuna') as pbar:
    def callback(study, trial):
        pbar.update(1)
        pbar.set_postfix({'mejor': f'{study.best_value:.4f}'})
    study.optimize(objective, n_trials=PARAM['n_trials'], callbacks=[callback])

print(f'\nOptuna finalizado.')
print(f'   Trials totales: {len(study.trials)}')
print(f'   Mejor {PARAM["metrica"]} (nivel): {study.best_value:.4f}')
print(f'   Mejores hiperparametros: {study.best_params}')


## 7) Guardar resultados

Guarda `tipo_target` para que `04_` reconstruya solo.


In [ ]:
resultado = {
    'experimento':      PARAM['experimento'],
    'modo_agrupacion':  PARAM['modo_agrupacion'],
    'metrica':          PARAM['metrica'],
    'mejor_valor':      study.best_value,
    'n_trials_total':   len(study.trials),
    'esquema_val':      PARAM['esquema_val'],
    'tipo_target':      TIPO_TARGET,
    'objective_lgbm':   PARAM['objective_lgbm'],
    'regularizacion':   PARAM['regularizacion'],
    'decay_recencia':   PARAM['decay_recencia'],
    'features':         FEATURES,
    'cat_features':     CAT_FEATURES,
    'hiperparametros':  study.best_params,
}

with open(PARAM['path_output'], 'w') as f:
    json.dump(resultado, f, indent=2)

print(f'Guardado: {PARAM["path_output"]}')
print(json.dumps(resultado, indent=2))

# Backup del study al bucket (el .db local se pierde si se destruye la VM)
db_local  = PARAM['path_storage'].replace('sqlite:///', '')
db_bucket = str(DIR_OUT / f"z303_optuna_{PARAM['modo_agrupacion']}.db")
shutil.copy(db_local, db_bucket)
print(f'Backup del study: {db_bucket}')


## 8) Visualizacion (opcional)


In [ ]:
try:
    import optuna.visualization as vis
    vis.plot_param_importances(study).show()
except Exception as e:
    print(f'Visualizacion no disponible: {e}')

df_trials = study.trials_dataframe()
print(df_trials[['number', 'value', 'state']].sort_values('value').head(10))
